# Sprint 4 · Reproducible operational handoff

Optional local companion to the [Campus journey](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/README.md). Keep your source/terminal project and observations. Independent cells require your own attempt before collapsed feedback. Figures come from executed synthetic reference data and never replace your raw evidence.

In [ ]:
from pathlib import Path
import json
import os
from linden_mlops import Workspace, batch_predict, reproduce
from linden_mlops.project import Project
from linden_mlops.lifecycle import ROOT, MODEL_NAME
from linden_mlops.operations import write_incident
from linden_mlops.release import verify_release
from linden_mlops.recovery import verify_bundle

opening_directory = Path.cwd().resolve()
if (opening_directory / "pyproject.toml").is_file():
    expected_root = opening_directory
elif opening_directory.name == "notebooks" and (opening_directory.parent / "pyproject.toml").is_file():
    expected_root = opening_directory.parent
else:
    expected_root = None
if expected_root is None or ROOT.resolve() != expected_root:
    raise RuntimeError("Choose this clone's .venv and start the notebook in its clone root or notebooks folder; the imported application must belong to that clone.")
os.chdir(ROOT)
project = Project()
workspace = Workspace(project.tracking)
print("Application source / working directory:", ROOT, Path.cwd())
print("Saved project:", project.work)

## A-S4-01 · What full reproducibility requires

🏷
# What full reproducibility requires

Linden Facilities can change a model in response to evidence. Another engineer now needs to rebuild the selected result without the original installation. Which dataset should they use if the current model was trained from feedback rather than the original historical snapshot?

## A fitted artifact and a recipe serve different purposes

A **fitted artifact** contains the learned scaler and model used for prediction. A **training recipe** records the ingredients and conditions needed to build it again. Loading the fitted artifact proves it can be used; fitting a new model from the recipe tests reproducibility. Missing model bytes can block loading while intact ingredients still permit retraining. A complete handoff retains both.

| Ingredient | What to identify | Why it matters |
|---|---|---|
| Code | Source bundle and content revision | A similar function name can hide changed behavior |
| Data | Exact selected training/validation snapshot and hash | Feedback-trained models need their feedback data |
| Configuration | `alpha`, range guard and logging settings | Defaults can differ between installations |
| Environment | Exact Python patch, packages, platform and lock | Numerical and serialization behavior can change |
| Model | Fitted bytes, source run and registered version | Connect selection to the actual predictor |

![Loading original model bytes and retraining from retained recipe ingredients are separate operations](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-4/01-what-full-reproducibility-requires/assets/what-full-reproducibility-requires__01__en.png)

This diagram describes required artifacts, not a completed recovery. Both the model and its recipe belong in the handoff.

## Two levels of proof

A **replay** retrains in a new store using the current source and environment. The final receiving proof adds restored source, a newly installed environment and a new store, with the original store unavailable. A successful replay is useful diagnosis; it is not that complete receiving proof.

Recovery must compare the same validation population as the original selected run. Absolute RMSE difference must be at most `0.000001` kWh under the [clean-state contract](https://github.com/richhiey/ml-llm-ops/blob/main/docs/clean-state.md). New run IDs, timestamps and version counters are expected. Batch delivery has its separate `24`-row, `60`-second requirement. The response comparison, in contrast, asks whether one model improves over another; exact reproduction asks whether the same recipe yields the same result.

## Keep operational evidence with the recipe

The portable export indexes `training-data.csv` when the run records that dataset artifact. It must match `manifest.json`'s data hash. Replacing feedback data with the original bundled CSV changes the recipe even if the schema matches. Declare the actual intended active version and source run in the application's release descriptor before export; the original experiment selection and later operational selection answer different questions.

Retain the policy, monitoring observation, candidate comparison and incident record alongside the model export. Those explain why selection changed. Checksums detect changes relative to the supplied index; they do not establish external authenticity or correctness of a learner's explanation.

## Try a diagnosis

A colleague supplies source, lock, configuration and a model, but substitutes the old historical data for the feedback snapshot. Explain which recovery invariant fails and why a similar RMSE would not repair it.

<details>
<summary>Check after recording your own decision and evidence</summary>

**Result you should get:** Data lineage fails: the selected feedback-trained recipe needs the exact recorded snapshot.

**How:** Compare the exported data bytes with the manifest hash and restore the canonical snapshot. Do not change the index or widen the numerical tolerance. A coincidentally similar result cannot substitute for matching ingredients.

</details>

## Learning outcome

Identify the code version, data snapshot, parameters/configuration, environment, and model artifact that must be traceable together for a training run to be reproducible, including the feedback dataset used by an operational response.

## Summary

A reproducible handoff carries both a usable fitted model and the ingredients for retraining it. Feedback data belongs to that recipe whenever it changed the model.


In [ ]:
active = workspace.client.get_model_version_by_alias(MODEL_NAME, "active")
active_run_id = active.run_id
print("Intended active source:", active_run_id)
print("Original experiment source:", project.selected()["run_id"])

## A-S4-02 · Integrating pipeline, tracking, and registry

✂️
# Integrating pipeline, tracking, and registry

Linden Facilities has training, registry and operational-response records. Another engineer needs a release that identifies the model actually intended for service and the recipe behind it. A plausible run ID or collection of successful files is insufficient.

## Start with an evidence inventory

Continue the same clone and cumulative workspace. Begin `work/HANDOFF.md` with criterion, existing record, observed or missing state, dependency and next check. Reuse the explained extension, distinguishing tests and original experiment comparison. Keep the historical five-run choice distinct from the operationally active model: a justified response can select a feedback-trained version without rewriting that experiment.

| Criterion | Question your evidence must answer |
|---|---|
| Connected execution | Which run trained the intended registered artifact? |
| Explained extension | What application behavior did you add and how do its tests distinguish correct from incomplete behavior? |
| Response | Which signal, labels and candidate comparison justified the action? |
| Delivery | Which intended version produced a complete, ordered planner file within the brief? |
| Reproducibility | Can the exact selected recipe be replayed, then received from fresh source/environment/state? |

Order the remaining checks by dependency. Keep missing evidence visible; a planned command is not a completed result.

## Your task — declare a usable release

Create `work/release.json` as the application's release descriptor. This file will be consumed by export; it is not a worksheet invented for this lesson. Use the actual intended active artifact and the job that served it.

| Required field | Value to establish from the application |
|---|---|
| `model_name` | Registered model name, here `dev.linden-demand` |
| `version` | Actual intended active version as a string |
| `run_id` | That registered artifact's actual source run |
| `batch_name` | Existing job name without the extension, such as `response-batch` |

Inspect the registry and raw run/job records before writing values. Do not substitute the matrix winner by habit. A source run and batch from different models do not become consistent merely because their predictions look similar.

Use `verify_release(workspace, path)` from the [release source](https://github.com/richhiey/ml-llm-ops/blob/main/src/linden_mlops/release.py) to check your descriptor against approved/current model state, source lineage and the actual batch CSV/JSON. Read the failure and repair your declared selection or genuinely missing evidence. Do not edit raw model, run or job records to make a descriptor pass. Preserve an unsuccessful attempt when it reveals a mismatch.

The release also requires the training run, serving job and current application source to agree. If you change application code after fitting, deliberately revalidate the intended recipe in a new run/version and produce a fresh named batch before export. Preserve the earlier records; do not relabel an old run as current or rerun a parameter search merely to repair lineage.

The application can record/check additional `config`, `data_sha256`, `code_revision` and `batch_input_sha256` fields. Inspect those actual values; a field copied from another release must be rejected. `write_release` provides an application-generated descriptor when you later automate the same operation, but your first task is to establish the correct identities yourself.

## Replay the release's actual source

Resolve the intended version using the [registry/API interfaces](https://github.com/richhiey/ml-llm-ops/blob/main/docs/api-reference.md). Predict which recipe invariants should match before invoking `reproduce(workspace, release_run_id, new_destination)` from the [lifecycle source](https://github.com/richhiey/ml-llm-ops/blob/main/src/linden_mlops/lifecycle.py). The source ID must be the one in your validated release descriptor.

A feedback-trained run retains `training-data.csv` among its reproduction artifacts. Replay consumes those bytes. It refuses changed source/environment and never tunes from final rows. If a retained run belongs to older application code, restore its exported source separately; do not edit a manifest to claim the current checkout is identical.

Open `reproduction-check.json`, inspect the new raw validation run and independently calculate the absolute RMSE difference. It must be at most `0.000001` kWh on the same validation population. Record the new run ID. This replay uses a new store in the current source/environment. A fresh-source/fresh-environment receiving proof remains a separate boundary.

## Working with documentation

The [release source](https://github.com/richhiey/ml-llm-ops/blob/main/src/linden_mlops/release.py) defines descriptor fields and checks. The [recovery runbook](https://github.com/richhiey/ml-llm-ops/blob/main/docs/recovery.md) explains replay versus clean receiving. Use those contracts to choose arguments; do not ask an assistant to invent a missing interface.

## Hints

<details><summary>Hint 1 — Follow the artifact backward</summary>

Start with the current intended registered version, then its raw source run, training snapshot and configuration. Keep the old experiment decision as historical evidence.

</details>
<details><summary>Hint 2 — Follow the job forward</summary>

The descriptor must name a real job whose version and source agree with the release. If you have only stale output, run a distinctly named new batch from the intended version; do not alter an old job's metadata.

</details>

## Checkpoints

Your inventory has evidence or a bounded next action for every criterion. `release.json` describes one approved/current artifact and its actual output. The selected snapshot/configuration/source/environment support replay within tolerance. Export will read this same descriptor, so the work carries forward into the actual handoff.

<details><summary>Check after creating the release and inspecting your replay</summary>

**Result you should get:** a valid application release descriptor, a dependency-ordered handoff inventory and a new-store replay of its exact recorded recipe. The original experiment decision may differ from the operational release without either record being rewritten.

**How:** join version → source run → data/configuration → actual job and inspect every supplied descriptor field against those records. Calculate the repeated validation-score delta independently. An equal score cannot compensate for different data or code. If verification refuses a mismatched field or stale job, preserve that failure and repair the actual declaration/evidence before export.

</details>

## Learning outcome

Scope the remaining handoff work, connect pipeline, tracking, registry and operational-response evidence, then replay the selected run from its recorded source, data, configuration and environment within the declared result tolerance.

## Summary

Your remaining work is scoped around a real release. Its model, run, data and serving record form one inspectable chain, and its recorded recipe has a replay result while the broader clean receiving proof remains to be established.


In [ ]:
# Start/extend project.work / "HANDOFF.md" with evidence, dependencies and bounded next actions.
# Author project.work / "release.json" from actual model/version/source/job records, then verify it.
# Replay that exact source into a new destination; compare raw lineage and calculate RMSE delta.
pass

<details><summary>Review after your own attempt</summary>

**Result you should get:** evidence matching the lesson checkpoints, with an action justified by raw records.

**How:** use the collapsed feedback in the lesson above to inspect your own result. Correct missing or contradicted evidence before advancing; do not substitute a copied reference verdict.

</details>

## A-S4-03 · Proving the selected serving path uses the active version

✂️
# Proving the selected serving path uses the active version

Linden Facilities has a selected operational model and evidence of why it was chosen. The next planner batch must actually load that version. An old output file cannot establish what a job after the response will serve.

## Inspect the observed response first

Continue the same cumulative source and workspace. Reuse your observed drift-trigger-response cycle when its source, policy, inputs and selected model still match. There is no requirement to retrain again merely to create a new report. If an element is missing, complete that bounded part using the already taught [operational interfaces](https://github.com/richhiey/ml-llm-ops/blob/main/docs/operations.md).

Before acting, write the signal, threshold and action supported by the retained evidence. Distinguish the unlabeled shift from the later independent labeled observation. Identify the candidate's training rows and the validation IDs used for both model errors. Explain why the decision was to promote, retain or recover, and what evidence would have justified a different response. Keep the original and feedback final rows outside this decision.

If your event concerns an already active unsuitable version, use the taught rollback route and a suitable approved replacement. Rejecting a candidate that was never active is not rollback. Retain failed comparisons instead of deleting an inconvenient result.

## Your task

1. Inventory the before/after model versions, actual source run, policy/report/data hashes and comparison decision. Separate observed facts from a proposed next action.
2. Predict the version that a new inference job should load. Invoke `batch_predict` with an explicit workspace and a distinct output name for the response, using the [API reference](https://github.com/richhiey/ml-llm-ops/blob/main/docs/api-reference.md).
3. Inspect the raw CSV and job metadata independently: `24` unique IDs in input order, finite predictions, matching input hash, matching intended run/version, and complete job time within `60` seconds. Calculate throughput as rows divided by elapsed seconds. This measures delivery, not training/setup time or business cost.
4. Prepare a separate invalid-input copy, predict the result, invoke the same application route with the intended workspace explicitly, and retain the rejection. Valid input must still succeed; rejection alone is not evidence of a useful pipeline.
5. Start the incident note in the existing `HANDOFF.md`: observation/action times, affected version, chosen response, resulting state, verification and one follow-up owner.

Your useful output is the complete planner CSV and its job record. Save the small independent CSV/metadata inspection in your own operational check script so another engineer can repeat it on a new job. Give it the input/output paths; do not hard-code a lesson ID or accept a report's `passed` field as its result.

A supplied synthetic intake stays labeled as such. Use your own observed run/version IDs; a reference picture or passing helper flag cannot replace the raw evidence.

## Hints

<details><summary>Look for the job after the decision</summary>

The alias describes selection. A fresh job's `resolved_version` and source `run_id` show which model that job loaded. Old output values can look plausible even when they came from a different version.

</details>

## Checkpoints

One observed operational cycle is connected to a newly verified batch from its resulting intended version. Contract rejection evidence is retained separately. Your explanation distinguishes distribution change, measured error, response acceptance and delivery correctness.

<details><summary>Check after recording the action and inspecting the output</summary>

**Result you should get:** the new job and current alias identify the intended artifact, with ordered finite output inside the delivery budget. The separate invalid-input probe rejects before a successful output is claimed.

**How:** compare raw registry/run/job records, inspect every output row against input IDs, and independently check values and elapsed time. If the job is stale or resolves another version, retain the mismatch, repair selection or invocation and execute a new job. Correct any incident claim of rollback when the candidate was never active.

</details>

## Learning outcome

Exercise a bounded drift-trigger-response cycle in the cumulative system, justify the response using the declared policy, and verify that a fresh inference job uses the resulting intended active model version and meets the inference brief.

## Summary

The handoff now includes proof that the intended version served after the operational decision, alongside the evidence explaining that response.


In [ ]:
# Connect your actual policy/monitor/candidate decision to the intended resulting active version.
# Produce a fresh distinctly named batch; inspect full IDs/order/finiteness/input hash/time yourself.
# Retain a separate invalid-input rejection and add observed response/verification to the incident.
pass

<details><summary>Review after your own attempt</summary>

**Result you should get:** evidence matching the lesson checkpoints, with an action justified by raw records.

**How:** use the collapsed feedback in the lesson above to inspect your own result. Correct missing or contradicted evidence before advancing; do not substitute a copied reference verdict.

</details>

## A-S4-04 · Setup, run, and expected-evidence instructions

✂️
# Setup, run, and expected-evidence instructions

Linden Facilities needs someone who has never used your checkout to run the pipeline. The handoff must carry both the recoverable recipe and instructions that explain where to start, what to execute and how to judge the result.

## Keep four locations distinct

A **portable export** is a copy of the selected run’s retained source, data, settings, environment record and fitted model. It can be read without the original MLflow database. A **checksum index**, `bundle.json`, records the expected content fingerprints of its files; checking those fingerprints detects a missing or changed copy.

| Place | Purpose |
|---|---|
| Original project | Existing source, `.venv`, selected validation run and `work/tracking` |
| Canonical export, such as `work/handoff-export` | Preserved reference package for the selected run |
| Receiving source, such as `../linden-received` | New folder restored from the package, with a newly installed `.venv` |
| Receiving store, such as `work/received` inside that source | New training, registry, audit and batch outputs |

The original model URI is a historical loading address. The receiver does not need that old address to remain reachable: the export retains the model files, and recovery trains/logs a new result locally.

![The portable recipe and a separate handoff document cross into restored source, a rebuilt environment and a new receiving store](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-4/04-setup-run-and-expected-evidence-instructions/assets/setup-run-and-expected-evidence-instructions__01__en.png)

The environment is rebuilt from the lock and recorded Python version; the diagram does not imply that the export contains an installed interpreter.

## Your task

Use the [export and receiving reference](https://github.com/richhiey/ml-llm-ops/blob/main/docs/recovery.md) to produce a receiver-ready `work/HANDOFF.md`. Continue the note you started; do not create another assignment report. Export the actual release once. Replace the paths if your descriptor or destination differs; the descriptor names the intended active source run and its real serving job:

```sh
uv run linden-mlops export --release work/release.json --destination work/handoff-export
```

If that export already exists, verify its recipe and reuse it instead of overwriting it. Compare its `manifest.json` source run with `release.json` and the actual intended active version before reuse. Training, serving and current source must agree at release; a source change after fitting needs a deliberately revalidated run/version and fresh batch before another export. An export of an older matrix winner cannot stand in for a later feedback-selected release. Neither a final observation nor a convenient new baseline is the intended recipe.

Explain the three operations in your instructions. `export` freezes retained run files into a package. `restore-source` unpacks its code/data/lock into a new folder. `reproduce-export` reads the exported recipe, retrains there into a new store, then logs/registers/selects the result and exercises batch inference. Choose exact receiving commands from the reference and replace every placeholder with your locations.

Your runbook must provide:

- The validated release descriptor and its selected source run, source-content revision, snapshot hash and recorded config: actual `alpha`, guard, autolog setting and validation partition. `reproduce-export` reads these from the manifest; do not invent extra model flags or substitute default `alpha=1`.
- The working directory and absolute package location for each command group. Include `uv run pytest tests/test_batch_input.py tests/test_spine.py::test_contract_extension_and_unchanged_pipeline -q` in the restored source so the added planned-batch stage is checked as well as the original guard. Read `manifest.json` → `environment.python` and install that **exact Python patch** with `uv sync --locked --python RECORDED_PATCH`. A matching `3.12` family alone may choose a different patch. Package versions, lock, operating system and architecture must also agree for this reproduction claim.
- A genuinely new source folder, newly installed environment and absent receiving-state path. Explain how you will temporarily make the original store unavailable, preserve it and restore it to its exact original location even if the test fails. Use the reference’s reversible isolation pattern; keep the export available.
- Expected output locations and checks: lineage fields that must agree, absolute validation RMSE difference at most `0.000001` kWh, newly resolved model/source identity, `24` finite predictions in input order and complete delivery within `60` seconds. New IDs, timestamps, paths and version counters are allowed.

An **invariant** is a field that must remain equal between the original recipe and receiving execution. A matching score with a different data fingerprint fails that rule. Exact-environment reproduction is also different from deliberately investigating another platform.

The automatic export does **not** include `work/HANDOFF.md` or your arbitrary working notes. Include this runbook beside the export in the handoff, with the extension’s changed logic/line-by-line explanation and relevant probe/diff evidence copied or quoted from your existing work. Use package-relative filenames for those supplementary records, so they remain available without the original checkout. Do not add them to or rewrite an existing checksum index; distinguish supplementary explanation from indexed runtime inputs.

## Working with documentation

The [recovery reference](https://github.com/richhiey/ml-llm-ops/blob/main/docs/recovery.md) includes `verify_bundle` for read-only integrity inspection and the reversible store-isolation pattern. Read it before the receiving investigation. Diagnose in order: package integrity, recorded source/data/config/environment, repeated evaluation, then delivered output. Later checks have not been tested when an earlier boundary stops execution.

For installation behavior, consult [uv locked synchronization](https://docs.astral.sh/uv/concepts/projects/sync/) and [Python version requests](https://docs.astral.sh/uv/concepts/python-versions/). Add recovery instructions for an existing destination, wrong interpreter, changed source or missing artifact. Preserve the first failure instead of editing a record to make it pass.

## Hints

<details><summary>Hint — Read as the receiver</summary>

For each command, trace every input to the transferred package, supplementary explanation or declared clean machine. If a path exists only in your original workspace, repair that dependency before running.

</details>

## Checkpoints

Your runbook includes resolved paths, exact setup/execution and acceptance criteria. The explained extension travels with it. The next clean receiving attempt will test this same runbook; use that one attempt as both documentation feedback and execution proof.

## Check your work after attempting the task

<details><summary>Compare after drafting the receiving instructions</summary>

**Result you should get:** a handoff comprising a verified selected-run export plus portable instructions and extension explanation. It reconstructs new source/environment/state using the exact recorded patch, keeps the original store unavailable during receiving and restores it afterward. It names independent output checks as well as the helper report.

**How:** trace command inputs backward and expected outputs forward. Use `verify_bundle` to inspect indexed files; it does not train. Check the manifest rather than assume a reference alpha or Python patch. If a note, diff or model file is outside the handoff, include the necessary content without altering the canonical export. Resolve uncertainty by testing this runbook once, not by making a second report.

</details>

## Include a light incident record

Add a short section to the same `HANDOFF.md`; do not write a second project report. Use the observed cycle, not a hypothetical story. Include the trigger signal, units, threshold, window and affected version; observation/action/verification times; the chosen response and its rationale; the resulting active version and fresh batch; and one follow-up with an owner and success check. Describe safe containment or an unresolved condition honestly if recovery was not established.

`write_incident` in the [operations source](https://github.com/richhiey/ml-llm-ops/blob/main/src/linden_mlops/operations.py) can package the raw observations with your explanation and follow-up. It checks that the report/comparison agree and the recorded batch is newer than the comparison and matches the resulting version. It does not assess whether your prose explains the decision. Review that separately. A deliberately worse candidate should leave explicit failure evidence, not disappear from the narrative.

The application consumes and validates `release.json` during export. Confirm its actual source run and job rather than relying on an implicit selection. When exporting a known run directly, supply its observed `--run-id` explicitly; use `--state` as well for an isolated client store. Keep `training-data.csv` inside its indexed bundle and preserve policy/monitor/comparison/incident evidence beside that bundle. Explain how to restore the recorded source and exact environment using the [receiving runbook](https://github.com/richhiey/ml-llm-ops/blob/main/docs/recovery.md).

<details>
<summary>Check after recording your own decision and evidence</summary>

**Result you should get:** A compact incident record and portable runbook point to actual observations and the exact export that will be tested.

**How:** Check the timeline against report and audit timestamps, version identity against the job, and data identity against the manifest. If a claim says rollback but the candidate was never active, correct the action description. Add an owner and observable follow-up check rather than a generic reminder.

</details>

## Learning outcome

Produce reproducible setup and run instructions naming the selected recipe and expected receiving evidence, plus a short incident record linking an observed trigger, affected version, timeline, response, verified resulting state and one owned follow-up action.

## Summary

Your instructions now describe a handoff someone else can execute and judge. Linden’s selected recipe, explained extension and acceptance rules can travel without depending on your installed workspace.


In [ ]:
# Export your verified release descriptor using the actual app command or recovery API.
# Write receiver-ready HANDOFF.md with actual paths, exact Python patch, isolation and acceptance.
# Carry extension explanation and an observed incident with one owned observable follow-up.
pass

<details><summary>Review after your own attempt</summary>

**Result you should get:** evidence matching the lesson checkpoints, with an action justified by raw records.

**How:** use the collapsed feedback in the lesson above to inspect your own result. Correct missing or contradicted evidence before advancing; do not substitute a copied reference verdict.

</details>

## A-S4-05 · Proving the pipeline reproduces

✂️
# Proving the pipeline reproduces

Linden Facilities has the exported recipe and your runbook. The remaining question is whether those instructions are sufficient without your original tracking store or installed project state. This attempt tests the handoff itself.

## Starting point

Use the canonical selected-validation export, `HANDOFF.md` and supplementary extension evidence. A smaller replay created a new store in the existing installation; this test additionally requires restored source and a fresh environment. Keep the same recorded configuration and validation rows.

| Context | What belongs there |
|---|---|
| Original-project terminal | Prepare/verify the export; stop producer processes; isolate and later restore the original store |
| Receiving terminal in restored source | Install the recorded environment, run contract tests and inspect outputs |
| Receiving source/environment, invoked by isolation script | Execute `reproduce-export` once while the original store is offline |
| Receiving store | Inspect the new run, registered version, alias event, output and recovery report |

## Your task

The deliverable is a receiver that actually works from the transferred package. Follow your own runbook verbatim. Use the [receiving reference](https://github.com/richhiey/ml-llm-ops/blob/main/docs/recovery.md) only to investigate missing instructions, then record the correction. Restore the archived application source into a new folder and confirm `.venv` does not already exist there. The receiver needs the runtime, selected data and environment; it must not depend on lesson dispatchers or an assessment generator. Read the original manifest’s Python patch before the locked installation; install that exact version. Verify the imported `linden_mlops` path belongs to the restored source and the interpreter belongs to its new `.venv`.

Make the original store unavailable during the receiving command using the reference’s reversible isolation pattern. Keep the canonical export readable and preserve the store intact. Record its original/offline locations, the absence check during execution and its restoration afterward. Stop the producer’s MLflow server and other commands that might reopen the old path; a newly created replacement store would invalidate the isolation claim. Do not delete the store or copy its SQLite database into the receiver.

Run `uv run pytest tests/test_batch_input.py tests/test_spine.py::test_contract_extension_and_unchanged_pipeline -q` in the receiving source. This checks your added planned-batch stage and the original domain guard from transferred source. Then use your runbook’s isolation script to invoke `reproduce-export` once in that source/environment, pointing at the absolute canonical-export path and a **new** receiving destination. Do not execute it separately before running the wrapper. Recovery retrains the recorded validation recipe; it does not merely load an old model and declare success. A destination refusal protects earlier evidence: inspect it or choose a new destination instead of deleting a prior attempt.

Inspect the outputs rather than stopping at a successful command:

| Check | What establishes it |
|---|---|
| Clean starting conditions | Restored source path, newly installed `.venv`, absent output store and original-store isolation record |
| Recipe agreement | Source/data/config/partition/environment comparisons in `recovery-check.json`, checked against the original manifest |
| Repeated evaluation | Original/repeated validation RMSE and their absolute difference `<=0.000001` kWh |
| Connected lifecycle | New run → registered version → `active` → batch’s recorded version/source; approval tags and alias event |
| Actual delivery | Input hash, all `24` IDs once in order, finite predictions and complete batch within `60` seconds |

The tolerance compares repeated evaluation scores; it is not an allowed error for each prediction. IDs, timestamps, paths and local version counters can change. Job timing is measured anew against the deadline, not against its old duration.

`recovery-check.json` has a useful `passed` field, but that flag alone does not establish the clean-folder history or independently check prediction order/finiteness. Read the actual `batch.csv` and its matching `batch.json` alongside the planned input. Explain which facts you observed directly.

Record commands, observed comparisons and pass/fail in the same `HANDOFF.md`. If something diverges, preserve the failed command/output and identify the first layer that failed: integrity, recipe/environment, evaluation or delivery. Repair the affected package/instruction and retry only the failed receiving proof from another fresh destination. Do not rerun the five-run search or widen tolerance to hide a mismatch.

Before opening feedback, diagnose two hypothetical reports: one has an equal score with a different snapshot; the other has new IDs with identical recipe and an accepted score difference. State which can count as reproduction and why. No extra training is needed for this reasoning.

## Working with documentation

Use the [clean-state specification](https://github.com/richhiey/ml-llm-ops/blob/main/docs/clean-state.md), [recovery command/inspection reference](https://github.com/richhiey/ml-llm-ops/blob/main/docs/recovery.md) and [`reproduce_bundle`](https://github.com/richhiey/ml-llm-ops/blob/main/src/linden_mlops/recovery.py). A failed preflight can leave later stages untested; do not infer their outcome from the failure time.

## Hints

<details><summary>Hint 1 — Check identities before the number</summary>

A familiar filename or equal score cannot compensate for changed source/data/environment. Start with the failed layer, not a different alpha.

</details>
<details><summary>Hint 2 — Separate what the command reports from how you ran it</summary>

The report cannot show that you created a new environment or made the original store unavailable. Your setup/isolation observations establish those facts.

</details>

## Checkpoints

The receiving execution uses only its transferred inputs and new installation/store. The original store is restored intact afterward. Both lineage and score tolerance pass, and actual output/model/timing checks agree. Keep this successful evidence for the final checkpoint; do not repeat it solely for another lesson page.

## Check your work after attempting the task

<details><summary>Compare your receiving proof and the author reference after your attempt</summary>

**Result you should get:** a new execution from restored source, a fresh exact-patch locked environment and a new store, completed while the original store was unavailable. Recipe invariants agree, absolute validation RMSE difference is within `0.000001` kWh, and a new approved active version produces `24` aligned finite predictions within the deadline. An equal score with different data fails; new execution IDs with matching lineage are expected.

**How:** compare the original manifest, installation/isolation record, receiving report, raw new run/version/audit and actual output. The receiving version’s source must be the newly trained run, not the original producer run. Restore mismatched prerequisites or repair instructions before a new receiving attempt. Preserve the first failure and keep numeric agreement separate from claims about the machine setup.

![Author reference shows restored source, fresh environment and state with separate lineage, score-difference and delivered-output checks](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-4/05-proving-the-pipeline-reproduces/assets/recovery-checks__01__en.png)

This is a verified **author reference**, not your execution. It separates recorded recipe checks from independently inspected output and the setup/isolation evidence. The delta axis is absolute validation RMSE difference in kWh; the boundary is `0.000001` kWh. Batch timing covers delivery only. Your IDs and observed duration can differ. The [reference observations](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/figures/reference-observations.json) retain the actual measured values and scope; the [optional notebook](https://github.com/richhiey/ml-llm-ops/blob/main/notebooks/04-cumulative-handoff.ipynb) can inspect your own receiving records.

</details>

## Verify the selected feedback recipe

If the intended active model was retrained from labeled feedback, the export's indexed `training-data.csv` must match that run's recorded hash. The receiver retrains from those bytes rather than silently substituting `demand-v1.csv`. Inspect both original and receiving manifests and calculate the validation RMSE delta on the same recorded partition. Keep the operational decision evidence beside the export; reproduction verifies that selected recipe, not the causal explanation for drift.

A failed checksum must stop before training. Preserve the failed output, restore the changed file from the canonical export, verify the unchanged index and all indexed bytes, and rerun any proof whose prior conditions no longer hold. Do not rewrite a checksum to bless altered data. A changed snapshot can retain the same schema and still invalidate the handoff.

## Learning outcome

Re-run the completed pipeline from the declared clean state, verify its lineage and result against the tolerance declared in the clean-state verification specification, and document and diagnose any divergence.

## Summary

The handoff is now supported by an actual clean receiving execution. You can explain both what reproduced and which observations establish independence from the original installation and store.


In [ ]:
# Follow your own runbook from restored source, a fresh exact environment and a new state.
# Keep producer store unavailable during receiving, restore it, and inspect raw lineage/score/output.
# Preserve the first divergence and its diagnosed repair; a same-environment replay is insufficient.
pass

<details><summary>Review after your own attempt</summary>

**Result you should get:** evidence matching the lesson checkpoints, with an action justified by raw records.

**How:** use the collapsed feedback in the lesson above to inspect your own result. Correct missing or contradicted evidence before advancing; do not substitute a copied reference verdict.

</details>

## A-S4-06 · Reproducible pipeline deliverable

✂️
# Reproducible pipeline deliverable

Harbor Facilities receives your cumulative operational pipeline. The recipient needs to inspect why a model was selected, rebuild that exact recipe and recognise an unfamiliar receiving failure. Existing good evidence can be reused when its source and conditions still match.

## What you will work with

Continue the same project and preserve the original five-run selection, explained extension and lifecycle evidence. Choose the intended operational version explicitly, with its raw decision evidence. Use the [clean-state contract](https://github.com/richhiey/ml-llm-ops/blob/main/docs/clean-state.md) and [receiving runbook](https://github.com/richhiey/ml-llm-ops/blob/main/docs/recovery.md); the four local notebooks remain optional companions.

## Your task

Deliver one connected handoff containing the explained extension and its distinguishing tests; original experiment comparison; actual registry/active-source chain; and one observed monitoring-trigger-response cycle. The cycle needs a declared policy, an unlabeled observation, separate labeled evidence where retraining is justified, candidate evaluation on disjoint validation rows, and the resulting selection or recovery. Explain a drift-without-retraining alternative so an alert cannot be mistaken for permission to train.

Include the short incident record in the same handoff. Its signal, threshold and affected version must agree with raw reports; its timeline must distinguish detection, response and verification. State containment or unresolved conditions honestly. Name one follow-up owner and an observable success check.

Use your validated application release descriptor to export the exact intended run with its fitted artifact and actual training snapshot into `work/handoff-export`. Reuse a canonical export only if its source/data/configuration still match the intended release. If your package is elsewhere, provide that actual canonical path to the intake setup. Restore source into a new folder, install the recorded Python patch and locked environment, and use a new store while the original store is unavailable. Preserve the documented reversible isolation/restoration procedure. Independently inspect matching code/data/configuration/environment, validation RMSE delta at most `0.000001` kWh, and `24` finite ordered predictions within `60` seconds. Record new run/version IDs. A replay in the original environment does not satisfy this receiving boundary.

## Diagnose an unfamiliar damaged receiving copy

The exercise intake supplies `work/receiving-review/brief.txt` and a `candidate-bundle/` received copy of your actual release export. The copy has a bounded unfamiliar receiving fault; the canonical export remains intact. Use the [intake setup reference](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/exercises/README.md) if the incoming package has not been provisioned. Case preparation belongs to exercise setup, outside the application runtime. Your task begins with inspecting the received package, not manufacturing a failure.

Read the supplied brief and preserve the first failure. Diagnose the earliest failing layer before editing. Repair the copied package from the canonical export without changing its checksum index, model selection, thresholds or tolerance. Explain when earlier receiving evidence remains applicable and when a fresh execution is necessary. Keep the canonical export and original store intact.

## Final check

The inference evidence must be from the resulting intended version. The incident must describe your observed response, not the guided reference fixture or an invented learner request. Reproduction data must be the selected run's snapshot, including feedback when applicable. Take any separate reserved final observation only after the relevant configuration/selection decisions are fixed; never use it to tune the response or claim current-condition quality from historical rows.

<details>
<summary>Check after recording your own decision and evidence</summary>

**Result you should get:** The exact selected recipe passes clean receiving with matching lineage, the declared RMSE tolerance and the batch contract. The incident explains a real bounded response with raw evidence. The damaged copy fails indexed config integrity before training; restoring canonical bytes repairs that integrity without rewriting the index.

**How:** Join every claim to the original and receiving records. Distinguish a checksum repair from a demonstrated clean run. Reuse a previous full receiving proof only for the same canonical recipe and equivalent source/environment/state conditions; otherwise rerun it. If the incident lacks a fresh job or a reasoned response, repair that evidence instead of substituting a screenshot or a passing Boolean.

</details>

## Learning outcome

Deliverable: the cumulative pipeline reproduced from the declared clean state with traceable code/data/configuration/environment/model lineage, reproducible setup/run instructions and an explained extension, plus one evidenced drift-trigger-retraining or recovery cycle, fresh active-version inference evidence, and a short incident record with a follow-up action.

## Summary

The delivered pipeline includes an observed response and a reproducible recipe. Another engineer can inspect the incident and rebuild the selected result without relying on the original installation.


In [ ]:
# Deliver the actual connected release, explained extension, response/incident and clean proof.
# Inspect supplied candidate-bundle before repair, retain first failure and restore canonical bytes.
# Explain whether earlier receiving proof still applies or which execution must be repeated.
pass

<details><summary>Review after your own attempt</summary>

**Result you should get:** evidence matching the lesson checkpoints, with an action justified by raw records.

**How:** use the collapsed feedback in the lesson above to inspect your own result. Correct missing or contradicted evidence before advancing; do not substitute a copied reference verdict.

</details>